<a href="https://colab.research.google.com/github/lexie009/CBB6340/blob/main/A4_E1_Pubmed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import time
import xml.etree.ElementTree as ET

import requests

In [2]:
def get_pubmed_data(pmid):
    """Fetch one PubMed record and return its raw XML as a string."""
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {"db": "pubmed", "id": str(pmid), "retmode": "xml"}

    # Wait before every request, including when the previous request failed.
    time.sleep(0.4)
    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()
    return response.content.decode("utf-8")


def extract_abstract(xml_payload):
    """Return the primary abstract with section labels, if present."""
    root = ET.fromstring(xml_payload)
    sections = []

    for node in root.findall(".//Article/Abstract/AbstractText"):
        # itertext() includes text inside inline tags and after them.
        text = "".join(node.itertext()).strip()
        if text:
            label = node.get("Label")
            sections.append(f"{label}: {text}" if label else text)

    # If there are no abstract sections, this returns an empty string.
    return "\n\n".join(sections)

In [3]:
for pmid in [34807729, 32299786]:
    xml_payload = get_pubmed_data(pmid)
    abstract = extract_abstract(xml_payload)

    root = ET.fromstring(xml_payload)
    blocks = root.findall(".//Article/Abstract/AbstractText")

    print(f"\nPMID {pmid}")
    print("Number of AbstractText blocks:", len(blocks))
    print("Labels:", [block.get("Label") for block in blocks])
    print("Inline tags:", [child.tag for block in blocks for child in block])
    print("\nExtracted abstract:")
    print(abstract)

    if pmid == 34807729:
        # This record has one plain-text block, so compare the entire text.
        matches_xml = len(blocks) == 1 and abstract == (blocks[0].text or "").strip()
        print("\nFull abstract matches the original XML block:", matches_xml)


PMID 34807729
Number of AbstractText blocks: 1
Labels: [None]
Inline tags: []

Extracted abstract:
Coronavirus disease 2019 (COVID-19) caused by severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2) infection is a systemic disease affecting not only the lungs but also multiple organ systems. Clinical studies implicate that SARS-CoV-2 infection causes imbalance of cellular homeostasis and immune response that trigger cytokine storm, oxidative stress, thrombosis, and insulin resistance. Mathematical modeling can offer in-depth understanding of the SARS-CoV-2 infection and illuminate how subcellular mechanisms and feedback loops underpin disease progression and multiorgan failure. We report here a mathematical model of SARS-CoV-2 infection pathway network with cytokine storm, oxidative stress, thrombosis, insulin resistance, and nitric oxide (NO) pathways. The biochemical systems theory model shows autocrine loops with positive feedback enabling excessive immune response, cytokine

In [4]:
pmid_list = [
    34807729, 21943025, 33980688, 32299786, 20725509,
    41916485, 17245822, 31842501, 35064122,
]

print("PMID\t\tchemokines\tCOVID-19\tStatus")

for pmid in pmid_list:
    try:
        xml_payload = get_pubmed_data(pmid)
        abstract = extract_abstract(xml_payload)

        # Check that EFetch actually returned the requested article.
        root = ET.fromstring(xml_payload)
        returned_pmid = root.findtext(".//MedlineCitation/PMID")
        if returned_pmid != str(pmid):
            print(f"{pmid}\tN/A\t\tN/A\t\tRequested record not returned")
            continue

        if not abstract:
            print(f"{pmid}\tN/A\t\tN/A\t\tNo abstract")
            continue

        # Word boundaries match the exact terms, ignoring capitalization.
        chemokines = "Yes" if re.search(r"\bchemokines\b", abstract, re.IGNORECASE) else "No"
        covid = "Yes" if re.search(r"\bCOVID-19\b", abstract, re.IGNORECASE) else "No"
        print(f"{pmid}\t{chemokines}\t\t{covid}\t\tOK")

    except (requests.RequestException, ET.ParseError, UnicodeError) as error:
        print(f"{pmid}\tN/A\t\tN/A\t\t{type(error).__name__}")

PMID		chemokines	COVID-19	Status
34807729	Yes		Yes		OK
21943025	No		No		OK
33980688	Yes		Yes		OK
32299786	Yes		No		OK
20725509	No		No		OK
41916485	N/A		N/A		No abstract
17245822	No		No		OK
31842501	No		No		OK
35064122	No		Yes		OK
